# 特征空间分析

输出 `results/` 下的 `pc_association.csv`、`explained_variance.csv`、`rank_axis.csv` 和 `rank_axis/<model>.npy`。
先运行初始化格；其余各格只依赖它，顺序随意。路径在 `fmspace/paths.py`。

In [ ]:
import numpy as np
import pandas as pd

from fmspace.analysis import association, compare_with_pc1, l2, pca, rank_axis
from fmspace.data import load_metadata, patient_split
from fmspace.extract import load_features
from fmspace.models import MODELS
from fmspace.paths import DATA_DIR, FEATURE_DIR, RESULT_DIR

metadata = load_metadata(DATA_DIR)
split = patient_split(metadata, seed=0)
X = {name: l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"])) for name in MODELS}
pcas = {name: pca(X[name]) for name in MODELS}

## PC × 标签关联、解释方差

PCA 用全部 1,189 张拟合（不用标签）。

In [ ]:
assoc = pd.concat([association(pcas[n].transform(X[n]), metadata).assign(model=n) for n in MODELS])
variance = pd.concat(
    [pd.DataFrame({"model": n, "component": np.arange(1, 51), "ratio": pcas[n].explained_variance_ratio_}) for n in MODELS]
)
assoc.to_csv(RESULT_DIR / "pc_association.csv", index=False)
variance.to_csv(RESULT_DIR / "explained_variance.csv", index=False)

## rank axis

按 Sonthalia et al. (NeurIPS 2025) 的流程训练。`rank_rho`、`pc1_rho`、`diff` 都在 seed 0 的测试集上计算，`diff_low`、`diff_high` 为患者 bootstrap 95% 区间。

In [ ]:
y = metadata["eye_dr_level"].to_numpy()
test = (split == "test").to_numpy()
rows = []
for name in MODELS:
    r = rank_axis(X[name], y, split)
    np.save(RESULT_DIR / "rank_axis" / f"{name}.npy", r["weight"])
    score = X[name] @ r["weight"]
    pc1 = pcas[name].transform(X[name])[:, 0]
    rows.append(
        {"model": name, "lr": r["lr"], "weight_decay": r["weight_decay"], "val_rho": r["val_rho"],
         **compare_with_pc1(score[test], pc1[test], y[test], metadata["patient_id"][test])}
    )
summary = pd.DataFrame(rows)
summary.to_csv(RESULT_DIR / "rank_axis.csv", index=False)
summary